## Middleware

In [2]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

### Summarization middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older
context. Summarization is useful for the following:

• Long-running conversations that exceed context windows.

• Multi-turn dialogues with extensive history.

• Applications where preserving full conversation context matters.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import SystemMessage, HumanMessage

# Message Summarization

agent = create_agent(
    model=ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=os.getenv("GEMINI_API_KEY")
    ),
    checkpointer = InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=ChatGoogleGenerativeAI(
                model="gemini-2.5-flash",
                google_api_key=os.getenv("GEMINI_API_KEY")
            ),
            trigger = ("messages",10),
            keep = ("messages",4)
        )
        ]
)

In [4]:
config = {"configurable":{"thread_id":"test-1"}}

In [5]:
questions = [
    "What is 2+2?",
    "what is 3+3?",
    "what is 4+4?",
    "what is 5+5?",
    "what is 6+6?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='28a5f1b0-7777-4a41-a77a-05877a0821ec'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a060dd-5875-7c40-bf15-78f6777c815b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 29, 'total_tokens': 37, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 22}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='28a5f1b0-7777-4a41-a77a-05877a0821ec'), AIMessage(content='2 + 2 = 4', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a060dd-5875-7c40-bf15-78f6777c815b-0', tool_c

### Human in loop Middleware

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str) -> str:
    # Simulate reading an email by returning a dummy email content
    return f"Email content for {email_id}"

def send_email_tool(email_id: str, content: str) -> str:
    # Simulate sending an email by returning a confirmation message
    return f"Email sent to {email_id} with content: {content}"

In [7]:
agent = create_agent(
    model=ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        google_api_key=os.getenv("GEMINI_API_KEY")
    ),
    checkpointer = InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "reject","edit"],
                },
                read_email_tool:False ,
            }
        )
    ]
)

In [ ]:
config = {"configurable":{"thread_id":"test-approve"}}

# Step 1

result = agent.invoke(
    {"messages":[HumanMessage(content="Send Email to john@test.com with subject 'Hello' and the body 'How are you?  '")]},config
)

